In [ ]:
import pandas as pd
import numpy as np
import joblib
import folium
import branca.colormap as cm

In [ ]:
df = pd.read_csv("../data/processed/featured_uhi_v2.csv")
model = joblib.load("../outputs/lst_model_xgb_v3.pkl")

FEATURES = ["NDVI", "NDBI", "Elevation", "Population"]
df["LST_baseline"] = model.predict(df[FEATURES])
print("Baseline LST predicted")

In [ ]:
# Composite of heat hazard + population exposure.
# Rank-normalise each to 0-1 (handles Population's heavy skew), then combine.
df["heat_rank"] = df["LST_baseline"].rank(pct=True)
df["pop_rank"]  = df["Population"].rank(pct=True)

df["HVI"] = 0.5 * df["heat_rank"] + 0.5 * df["pop_rank"]

print(df[["LST_baseline", "Population", "HVI"]].describe())

In [ ]:
n_bins = 10
lat_edges = np.linspace(df["Latitude"].min(), df["Latitude"].max(), n_bins + 1)
lon_edges = np.linspace(df["Longitude"].min(), df["Longitude"].max(), n_bins + 1)
df["lat_bin"] = pd.cut(df["Latitude"], bins=lat_edges, labels=False, include_lowest=True)
df["lon_bin"] = pd.cut(df["Longitude"], bins=lon_edges, labels=False, include_lowest=True)
df["zone"] = df["lat_bin"].astype(int).astype(str) + "_" + df["lon_bin"].astype(int).astype(str)

zone_vuln = df.groupby("zone").agg(
    HVI          = ("HVI", "mean"),
    LST_baseline = ("LST_baseline", "mean"),
    Population   = ("Population", "mean"),
    Latitude     = ("Latitude", "mean"),
    Longitude    = ("Longitude", "mean")
).reset_index().sort_values("HVI", ascending=False).reset_index(drop=True)

zone_vuln.to_csv("../outputs/reports/zone_vulnerability.csv", index=False)
zone_vuln.head(10)

In [ ]:
colormap = cm.LinearColormap(
    colors=["#1a9850", "#ffffbf", "#fdae61", "#d73027", "#7f0000"],
    vmin=zone_vuln["HVI"].min(),
    vmax=zone_vuln["HVI"].max(),
    caption="Heat Vulnerability Index (heat + people)"
)

center = [df["Latitude"].mean(), df["Longitude"].mean()]
m = folium.Map(location=center, zoom_start=11, tiles="CartoDB positron")

cell_hvi = df.groupby(["lat_bin", "lon_bin"])["HVI"].mean().reset_index()
for _, c in cell_hvi.iterrows():
    i, j = int(c["lat_bin"]), int(c["lon_bin"])
    bounds = [[lat_edges[i], lon_edges[j]], [lat_edges[i + 1], lon_edges[j + 1]]]
    folium.Rectangle(
        bounds=bounds, weight=0, fill=True,
        fill_color=colormap(c["HVI"]), fill_opacity=0.6,
        popup=f"HVI: {c['HVI']:.2f}"
    ).add_to(m)

for rank, row in zone_vuln.head(5).iterrows():
    folium.Marker(
        location=[row["Latitude"], row["Longitude"]],
        popup=(f"Most vulnerable #{rank+1}<br>"
               f"Zone {row['zone']}<br>"
               f"HVI: {row['HVI']:.2f}<br>"
               f"LST: {row['LST_baseline']:.1f} C<br>"
               f"Pop index: {row['Population']:.1f}"),
        icon=folium.Icon(color="red", icon="users", prefix="fa")
    ).add_to(m)

colormap.add_to(m)
m.save("../outputs/aurangabad_vulnerability_map.html")
print("Saved -> outputs/aurangabad_vulnerability_map.html")
m